# xRouteBench — Dataset Study for Building a Router

**Goal:** understand the routing problem before training a router.
A router picks one model from an 18-model candidate pool per query,
trading off **performance** vs **price cost**.

Running on the **`route_f`** kernel.

In [ ]:
import sys
sys.path.insert(0, ".")
from data.load_xroutebench import (
    list_configs, schema, load, build_price_table, row_cost,
)
from collections import Counter, defaultdict
import statistics, hashlib

print("configs:", len(list_configs()))
for c in list_configs():
    print("  -", c)

## 1. Routing-data config (llmrouter_generic)

Each row is one `(query, candidate model)` execution with a score.
Every query was run against all 18 models → `18 × #queries` rows.

In [ ]:
ds = load("llmrouter_generic", "train", stream=False)
print("rows:", len(ds), "| unique queries:", len(ds) // len(set(ds["model_name"])))

print("\n=== tasks ===")
for t, c in Counter(ds["task_name"]).most_common():
    print(f"  {t:18s} {c:6d}")

print("\n=== scoring metrics ===")
for m, c in Counter(ds["metric"]).most_common():
    print(f"  {m:12s} {c:6d}")

perf = ds["performance"]
print("\n=== performance stats ===")
print("  mean %.4f | median %.4f | min %.3f | max %.3f" % (
    statistics.mean(perf), statistics.median(perf), min(perf), max(perf)))
tok = ds["token_num"]
print("  tokens: mean %.0f | median %.0f | max %d" % (
    statistics.mean(tok), statistics.median(tok), max(tok)))

## 2. Candidate pool & pricing (llm_candidates)

18 models with `input_price_per_1m` / `output_price_per_1m` (USD / 1M tokens).

In [ ]:
cand = load("llm_candidates", "train", stream=False)
price = build_price_table(cand)
for r in cand:
    print("  %-28s in=$%4.2f/M out=$%4.2f/M  %s" % (
        r["model_name"], r["input_price_per_1m"], r["output_price_per_1m"], r["service"]))

# price spread
ins = [r["input_price_per_1m"] for r in cand]
print("\ninput price range: $%.2f–$%.2f per 1M (%.1f× spread)" % (
    min(ins), max(ins), max(ins)/min(ins)))

## 3. The routing target

For each query, which model is **best** (highest performance), and which is **cheapest**?
This defines the heads of the performance–cost tradeoff.

In [ ]:
# group executions by query (task_name + full query hash)
by_query = defaultdict(list)
for r in ds:
    key = (r["task_name"], hashlib.md5(r["query"].encode()).hexdigest())
    by_query[key].append(r)

best_ct = Counter(); cheap_ct = Counter()
best_perf, cheap_perf = [], []
for rows in by_query.values():
    best = max(rows, key=lambda r: r["performance"])
    cheap = min(rows, key=lambda r: row_cost(r, price))
    best_ct[best["model_name"]] += 1
    cheap_ct[cheap["model_name"]] += 1
    best_perf.append(best["performance"]); cheap_perf.append(cheap["performance"])

n = len(by_query)
print("=== BEST model per query (routing target) ===")
for m, c in best_ct.most_common():
    print("  %-28s %5d  %5.1f%%" % (m, c, 100*c/n))
print("\n=== CHEAPEST model per query ===")
for m, c in cheap_ct.most_common(6):
    print("  %-28s %5d  %5.1f%%" % (m, c, 100*c/n))
print("\n=== performance: best vs cheapest (per query) ===")
print("  best    : mean %.3f | median %.3f" % (statistics.mean(best_perf), statistics.median(best_perf)))
print("  cheapest : mean %.3f | median %.3f" % (statistics.mean(cheap_perf), statistics.median(cheap_perf)))

## 4. Interpretation — what this means for the router

- **A cheap model wins surprisingly often**: `gpt-oss-20b` ($0.05/M) is cheapest
  for ~86% of queries, and its performance is close-ish to the best model on
  average. A good router should thus **default to cheap, escalate to expensive.**
- **`qwen2.5-7b-instruct` / `gemma-2-9b-it`** are the workhorse *best* models,
  both cheap — so a cost-aware router can capture most quality at low cost.
- **Embeddings matter**: only `query` text is given (embeddings are not shipped),
  so we must embed queries ourselves (any sentence encoder) to build features.
- **Metric-aware**: tasks use different metrics (`em_mc`, `GSM8K`, `code_eval`,
  `f1_score`, `MATH`). Normalize performance per task/metric before training.

**Recommended next step:** build `embedding → (task features) → pick model`
classifier, optimizing a composite reward `α·norm(performance) − β·norm(cost)`.